In [1]:
import os
import json
import numpy as np
import pandas as pd
import wfdb
import pydicom
import warnings
import logging
from pydicom.pixels import convert_color_space
from PIL import Image
from tqdm.auto import tqdm

# Silence pydicom warnings about truncated DICOMs (just noise in progress bar)
warnings.filterwarnings('ignore', category=UserWarning, module='pydicom')
logging.getLogger('pydicom').setLevel(logging.ERROR)

# --- PATHS ---
DATA_DIR    = r"C:\Users\anwme\Desktop\Datasets\Physionet"
ECG_DIR     = os.path.join(DATA_DIR, "Physionet-ECG")
ECHO_DIR    = os.path.join(DATA_DIR, "Physionet-ECHO")

# CSV should be in the preprocessing folder (the env folder). Move it there if needed.
PROJECT_DIR = r"C:\Users\anwme\Desktop\preprocessing"
CSV_PATH    = os.path.join(PROJECT_DIR, "final_dataset.csv")
CACHE_DIR   = os.path.join(PROJECT_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"ECG dir:   {ECG_DIR}")
print(f"           exists: {os.path.isdir(ECG_DIR)}")
print(f"Echo dir:  {ECHO_DIR}")
print(f"           exists: {os.path.isdir(ECHO_DIR)}")
print(f"CSV:       {CSV_PATH}")
print(f"           exists: {os.path.isfile(CSV_PATH)}")
print(f"Cache out: {CACHE_DIR}")

ECG dir:   C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECG
           exists: True
Echo dir:  C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO
           exists: True
CSV:       C:\Users\anwme\Desktop\preprocessing\final_dataset.csv
           exists: True
Cache out: C:\Users\anwme\Desktop\preprocessing\cache


In [2]:
# Load CSV
df = pd.read_csv(CSV_PATH)
print(f"Rows in CSV: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()

# Peek at the path columns to understand the CSV's path format
print("=== CSV path samples (first row) ===")
print(f"ecg_path:      {df['ecg_path'].iloc[0]}")
print(f"echo_path_ED:  {df['echo_path_ED'].iloc[0]}")
print(f"echo_path_Mid: {df['echo_path_Mid'].iloc[0]}")
print(f"echo_path_ES:  {df['echo_path_ES'].iloc[0]}")
print()

# Inspect your actual ECG folder layout — is it flat or nested?
print("=== ECG folder — first 5 items ===")
ecg_contents = os.listdir(ECG_DIR)[:5]
for item in ecg_contents:
    full = os.path.join(ECG_DIR, item)
    kind = "DIR " if os.path.isdir(full) else "FILE"
    print(f"  [{kind}] {item}")

print(f"\nTotal items in ECG folder: {len(os.listdir(ECG_DIR))}")

# If it's nested, go one level deeper to see what's inside
first_ecg = os.path.join(ECG_DIR, os.listdir(ECG_DIR)[0])
if os.path.isdir(first_ecg):
    print(f"\nContents of first ECG subfolder ({os.listdir(ECG_DIR)[0]}):")
    for item in os.listdir(first_ecg)[:5]:
        full = os.path.join(first_ecg, item)
        kind = "DIR " if os.path.isdir(full) else "FILE"
        print(f"  [{kind}] {item}")

print()

# Same check for Echo folder
print("=== ECHO folder — first 5 items ===")
echo_contents = os.listdir(ECHO_DIR)[:5]
for item in echo_contents:
    full = os.path.join(ECHO_DIR, item)
    kind = "DIR " if os.path.isdir(full) else "FILE"
    print(f"  [{kind}] {item}")

print(f"\nTotal items in ECHO folder: {len(os.listdir(ECHO_DIR))}")

first_echo = os.path.join(ECHO_DIR, os.listdir(ECHO_DIR)[0])
if os.path.isdir(first_echo):
    print(f"\nContents of first ECHO subfolder ({os.listdir(ECHO_DIR)[0]}):")
    for item in os.listdir(first_echo)[:5]:
        full = os.path.join(first_echo, item)
        kind = "DIR " if os.path.isdir(full) else "FILE"
        print(f"  [{kind}] {item}")

Rows in CSV: 5448
Columns: ['subject_id', 'study_id_ecg', 'study_id_echo', 'ecg_time', 'study_datetime', 'time_diff', 'ecg_path', 'echo_path_ED', 'echo_path_Mid', 'echo_path_ES', 'arrhythmia', 'arrhythmia_coarse', 'arrhythmia_labels', 'structural', 'structural_coarse', 'structural_labels', 'split']

=== CSV path samples (first row) ===
ecg_path:      files/p1997/p19971226/s40000903/40000903
echo_path_ED:  files/p19/p19971226/s90266158/90266158_0001.dcm
echo_path_Mid: files/p19/p19971226/s90266158/90266158_0036.dcm
echo_path_ES:  files/p19/p19971226/s90266158/90266158_0076.dcm

=== ECG folder — first 5 items ===
  [DIR ] p10024120
  [DIR ] p10024736
  [DIR ] p10036086
  [DIR ] p10040025
  [DIR ] p10046592

Total items in ECG folder: 1714

Contents of first ECG subfolder (p10024120):
  [DIR ] s48734398

=== ECHO folder — first 5 items ===
  [DIR ] p10024120
  [DIR ] p10024736
  [DIR ] p10036086
  [DIR ] p10040025
  [DIR ] p10046592

Total items in ECHO folder: 1714

Contents of first ECH

In [3]:
# Build absolute paths on your disk from CSV relative paths
def resolve_echo(csv_rel_path, base_dir):
    # 'files/p19/p19971226/s.../XXX.dcm' → base_dir/p19971226/s.../XXX.dcm
    parts = csv_rel_path.replace('\\', '/').split('/')
    # parts = ['files', 'p19', 'p19971226', 's90266158', '90266158_0001.dcm']
    return os.path.join(base_dir, *parts[2:])

def resolve_ecg(csv_rel_path, base_dir):
    # 'files/p1997/p19971226/s40000903/40000903' → base_dir/p19971226/s40000903/40000903
    parts = csv_rel_path.replace('\\', '/').split('/')
    return os.path.join(base_dir, *parts[2:])

# Apply to every row
df['ecg_full']  = df['ecg_path'].apply(lambda p: resolve_ecg(p, ECG_DIR))
df['echo_ED']   = df['echo_path_ED'].apply(lambda p: resolve_echo(p, ECHO_DIR))
df['echo_Mid']  = df['echo_path_Mid'].apply(lambda p: resolve_echo(p, ECHO_DIR))
df['echo_ES']   = df['echo_path_ES'].apply(lambda p: resolve_echo(p, ECHO_DIR))

# Sanity check: print the first resolved paths
print("=== First row resolved paths ===")
print(f"ECG:     {df['ecg_full'].iloc[0]}")
print(f"Echo ED: {df['echo_ED'].iloc[0]}")
print(f"Echo Mid: {df['echo_Mid'].iloc[0]}")
print(f"Echo ES: {df['echo_ES'].iloc[0]}")
print()

# Check if these files actually exist
row0 = df.iloc[0]
print(f"ECG .hea exists:   {os.path.isfile(row0['ecg_full'] + '.hea')}")
print(f"ECG .dat exists:   {os.path.isfile(row0['ecg_full'] + '.dat')}")
print(f"Echo ED exists:    {os.path.isfile(row0['echo_ED'])}")
print(f"Echo Mid exists:   {os.path.isfile(row0['echo_Mid'])}")
print(f"Echo ES exists:    {os.path.isfile(row0['echo_ES'])}")

=== First row resolved paths ===
ECG:     C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECG\p19971226\s40000903\40000903
Echo ED: C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158\90266158_0001.dcm
Echo Mid: C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158\90266158_0036.dcm
Echo ES: C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158\90266158_0076.dcm

ECG .hea exists:   True
ECG .dat exists:   True
Echo ED exists:    True
Echo Mid exists:   True
Echo ES exists:    True


In [4]:
def row_available(r):
    return (os.path.isfile(r['ecg_full'] + '.hea') and
            os.path.isfile(r['ecg_full'] + '.dat') and
            os.path.isfile(r['echo_ED']) and
            os.path.isfile(r['echo_Mid']) and
            os.path.isfile(r['echo_ES']))

print("Scanning all 5448 rows for file availability (local SSD, ~1 min)...")
df['available'] = df.apply(row_available, axis=1)
df_avail = df[df['available']].reset_index(drop=True)

print(f"\nAvailable (all 5 files present): {len(df_avail)} / {len(df)}")
print(f"\nSplit distribution (available only):")
print(df_avail['split'].value_counts())
print(f"\nstructural_coarse distribution (available only):")
print(df_avail['structural_coarse'].value_counts())

# Also show what's MISSING — helpful to confirm download is complete
missing = df[~df['available']]
if len(missing) > 0:
    print(f"\n⚠ {len(missing)} rows have missing files. Sample missing paths:")
    for i, r in missing.head(3).iterrows():
        for col in ['ecg_full', 'echo_ED', 'echo_Mid', 'echo_ES']:
            p = r[col] + ('.hea' if col == 'ecg_full' else '')
            if not os.path.isfile(p):
                print(f"  MISSING: {p}")
                break

# Save the filtered df for later cells (so we don't re-scan)
df_avail.to_csv(os.path.join(CACHE_DIR, 'df_avail.csv'), index=False)
print(f"\nSaved filtered dataframe to {CACHE_DIR}\\df_avail.csv")

Scanning all 5448 rows for file availability (local SSD, ~1 min)...

Available (all 5 files present): 5448 / 5448

Split distribution (available only):
split
train    3761
test     1124
val       563
Name: count, dtype: int64

structural_coarse distribution (available only):
structural_coarse
ischemic_heart_disease    2268
heart_failure             1339
normal                     948
cardiomyopathy             543
valve_disease              213
pericardial_disease        100
congenital                  37
Name: count, dtype: int64

Saved filtered dataframe to C:\Users\anwme\Desktop\preprocessing\cache\df_avail.csv


In [5]:
row0 = df.iloc[0]
echo_folder = os.path.dirname(row0['echo_ED'])
print(f"Looking inside: {echo_folder}")
print(f"Folder exists: {os.path.isdir(echo_folder)}\n")

if os.path.isdir(echo_folder):
    files_in_folder = sorted(os.listdir(echo_folder))
    print(f"Files present ({len(files_in_folder)} total):")
    for f in files_in_folder:
        print(f"  {f}")
    print(f"\nCSV expected files:")
    print(f"  {os.path.basename(row0['echo_ED'])}")
    print(f"  {os.path.basename(row0['echo_Mid'])}")
    print(f"  {os.path.basename(row0['echo_ES'])}")

Looking inside: C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158
Folder exists: True

Files present (3 total):
  90266158_0001.dcm
  90266158_0036.dcm
  90266158_0076.dcm

CSV expected files:
  90266158_0001.dcm
  90266158_0036.dcm
  90266158_0076.dcm


In [6]:
# Confirm Windows isn't hiding the files with case issues
import glob
pattern = os.path.join(echo_folder, "*")
all_files = glob.glob(pattern)
print(f"glob found {len(all_files)} files")
for f in all_files:
    print(f"  {f}")

glob found 3 files
  C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158\90266158_0001.dcm
  C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158\90266158_0036.dcm
  C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO\p19971226\s90266158\90266158_0076.dcm


In [7]:
from collections import Counter
# For each row in CSV, count how many of the 3 frames are present
def count_present(r):
    return sum(os.path.isfile(r[c]) for c in ['echo_ED', 'echo_Mid', 'echo_ES'])

df['n_echo_present'] = df.apply(count_present, axis=1)
print("Distribution of frames-present per study:")
print(df['n_echo_present'].value_counts().sort_index())
# Also check ECGs
df['ecg_present'] = df.apply(lambda r: os.path.isfile(r['ecg_full']+'.hea') and os.path.isfile(r['ecg_full']+'.dat'), axis=1)
print(f"\nECGs present: {df['ecg_present'].sum()} / {len(df)}")

Distribution of frames-present per study:
n_echo_present
3    5448
Name: count, dtype: int64

ECGs present: 5448 / 5448


In [8]:
# Check size distribution of downloaded echo files
import os
from pathlib import Path

sizes_kb = []
for dcm in Path(ECHO_DIR).rglob("*.dcm"):
    sizes_kb.append(dcm.stat().st_size / 1024)

import numpy as np
sizes = np.array(sizes_kb)
print(f"Total DICOM files: {len(sizes)}")
print(f"Size distribution (KB):")
print(f"  min:    {sizes.min():.1f}")
print(f"  median: {np.median(sizes):.1f}")
print(f"  mean:   {sizes.mean():.1f}")
print(f"  max:    {sizes.max():.1f}")
print(f"\nBuckets:")
print(f"  < 50 KB     : {(sizes < 50).sum()}      (suspicious — possibly truncated)")
print(f"  50-100 KB   : {((sizes >= 50) & (sizes < 100)).sum()}")
print(f"  100-500 KB  : {((sizes >= 100) & (sizes < 500)).sum()}")
print(f"  500KB-2MB   : {((sizes >= 500) & (sizes < 2048)).sum()}")
print(f"  2-10 MB     : {((sizes >= 2048) & (sizes < 10240)).sum()}")
print(f"  > 10 MB     : {(sizes >= 10240).sum()}")

Total DICOM files: 7093
Size distribution (KB):
  min:    37.5
  median: 3747.1
  mean:   3459.5
  max:    68302.4

Buckets:
  < 50 KB     : 4      (suspicious — possibly truncated)
  50-100 KB   : 955
  100-500 KB  : 2018
  500KB-2MB   : 52
  2-10 MB     : 3961
  > 10 MB     : 103


In [9]:
# Pick a random small file to verify it's valid DICOM
import pydicom
from pathlib import Path

small_files = [p for p in Path(ECHO_DIR).rglob("*.dcm") if p.stat().st_size < 100 * 1024][:3]
for p in small_files:
    try:
        dcm = pydicom.dcmread(str(p), stop_before_pixels=True)
        rows = getattr(dcm, 'Rows', '?')
        cols = getattr(dcm, 'Columns', '?')
        frames = getattr(dcm, 'NumberOfFrames', 1)
        modality = getattr(dcm, 'Modality', '?')
        print(f"{p.name:30s} | {p.stat().st_size/1024:>6.1f} KB | {rows}x{cols} x{frames} | {modality}")
    except Exception as e:
        print(f"{p.name:30s} | ERROR: {e}")

97850386_0066.dcm              |   95.2 KB | 708x1016 x1 | US
97850386_0148.dcm              |   87.9 KB | 708x1016 x1 | US
96993263_0002.dcm              |   73.2 KB | 708x1016 x1 | US
